In [9]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import numpy as np


In [10]:
page_number = 1
base_url = f"https://www.flipkart.com/search?q=smartphone&as=on&as-show=on&otracker=AS_Query_OrganicAutoSuggest_4_5_na_na_na&otracker1=AS_Query_OrganicAutoSuggest_4_5_na_na_na&as-pos=4&as-type=RECENT&suggestionId=smartphone&requestId=0d3788ae-4292-4222-b969-cf2f01a59412&as-searchtext=smart&page={page_number}"




In [11]:
headers = {
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3",
    "accept-encoding": "gzip",
    "accept-language": "en-US,en;q=0.9",
    "content-type": "application/x-www-form-urlencoded"
}

In [12]:
res = requests.get(base_url, headers=headers)
if res.status_code!=200:
    print(f"Failed to fetch page {page_number}. Status code: {res.status_code}")
else:
    print(f"Successfully fetched page {page_number}.")

soup = BeautifulSoup(res.text, "html.parser")
data= soup.find('div', attrs= {"class":"FF6GgS"})
print(data)


Successfully fetched page 1.
<div class="FF6GgS"><span>Page 1 of 577</span><nav class="iu0OAI"><a class="i2eZXn paN7Fu" href="/search?q=smartphone&amp;as=on&amp;as-show=on&amp;otracker=AS_Query_OrganicAutoSuggest_4_5_na_na_na&amp;otracker1=AS_Query_OrganicAutoSuggest_4_5_na_na_na&amp;as-pos=4&amp;as-type=RECENT&amp;suggestionId=smartphone&amp;requestId=0d3788ae-4292-4222-b969-cf2f01a59412&amp;as-searchtext=smart&amp;page=1">1</a><a class="i2eZXn" href="/search?q=smartphone&amp;as=on&amp;as-show=on&amp;otracker=AS_Query_OrganicAutoSuggest_4_5_na_na_na&amp;otracker1=AS_Query_OrganicAutoSuggest_4_5_na_na_na&amp;as-pos=4&amp;as-type=RECENT&amp;suggestionId=smartphone&amp;requestId=0d3788ae-4292-4222-b969-cf2f01a59412&amp;as-searchtext=smart&amp;page=2">2</a><a class="i2eZXn" href="/search?q=smartphone&amp;as=on&amp;as-show=on&amp;otracker=AS_Query_OrganicAutoSuggest_4_5_na_na_na&amp;otracker1=AS_Query_OrganicAutoSuggest_4_5_na_na_na&amp;as-pos=4&amp;as-type=RECENT&amp;suggestionId=smartpho

In [13]:
print("=============================")
pages = data.find_all('span')

print(pages[0].text)

tot_pages_count = pages[0].text.split()[-1]
print(tot_pages_count)

Page 1 of 577
577


In [14]:


prod_name = []
prod_rating = []
prod_price = []
prod_act_price = []
prod_page = []
prod_brand = []
prod_ram = []
prod_storage = []
battery_details = []
camera_details = []
prod_discount = []

range_counts = range(1, int(tot_pages_count)+1)
print(range_counts)
continue_5_pages_rows_count=-1
for page in range_counts:
    url1 = f"https://www.flipkart.com/search?q=smartphone&as=on&as-show=on&otracker=AS_Query_OrganicAutoSuggest_4_5_na_na_na&otracker1=AS_Query_OrganicAutoSuggest_4_5_na_na_na&as-pos=4&as-type=RECENT&suggestionId=smartphone&requestId=0d3788ae-4292-4222-b969-cf2f01a59412&as-searchtext=smart&page={page}"

    res1 = requests.get(url1)
    res1.status_code
    soup1 = BeautifulSoup(res1.text, "html.parser")

    print(f"============Page: {page}")
    print(f"URL--------------- {url1}")

    all_rows = soup1.find_all('div', attrs= {"class":"ZFwe0M row"})
    print(f"all rows count: {len(all_rows)}")

    if len(all_rows)==0:
        print(f"Page {page} has no product rows. Skipping to next page.")
        continue_5_pages_rows_count = continue_5_pages_rows_count+1
        if continue_5_pages_rows_count==5:
            print(f"5 consecutive pages have no product rows. Stopping the scraping process.")
            break
        continue

    for row in all_rows:
        continue_5_pages_rows_count = -1
        pname_ele = row.find('div', attrs= {"class":"RG5Slk"})
        prod_name.append(pname_ele.text[:50] if pname_ele else np.nan)

        brand = pname_ele.text.split()[0] if pname_ele else None
        prod_brand.append(brand)

        rating_ele = row.find('div', attrs= {"class":"MKiFS6"})
        prod_rating.append(rating_ele.text if rating_ele else np.nan)

        price_ele = row.find('div', attrs= {"class":"hZ3P6w DeU9vF"})
        prod_price.append(price_ele.text if price_ele else np.nan)

        act_price_ele = row.find('div', attrs= {"class":"kRYCnD gxR4EY"})
        prod_act_price.append(act_price_ele.text if act_price_ele else np.nan)

        discount = row.find('div', attrs= {"class":"HQe8jr"})
        prod_discount.append(discount.text if discount else np.nan)


        mobile_details = row.find_all('li', attrs= {"class":"DTBslk"})
        product_battery_details =[]
        product_camera_details =[]
        product_ram = np.nan
        product_storage = np.nan

        for detail in mobile_details:
            info = detail.text
            if "gb ram" in info.lower() and "gb rom" in info.lower():
                arr_memory_details = info.split("|") if info else np.nan
                ram_size = arr_memory_details[0].strip()
                storage = arr_memory_details[1].strip()

                product_ram = ram_size if ram_size else np.nan
                product_storage = storage if storage else np.nan



            elif "gb ram" not in info.lower() and "gb rom" in info.lower():     ##in this case only ROM data is available
               product_storage = info.strip() if info else np.nan


            elif "lithium" in info.lower() or "battery" in info.lower() or "mah" in info.lower():
                product_battery_details.append(info)

            elif "camera" in info.lower() or "MP" in info:
                product_camera_details.append(info)

        prod_ram.append(product_ram)
        prod_storage.append(product_storage)
        battery_details.append(", ".join(product_battery_details) if product_battery_details else np.nan)
        camera_details.append(", ".join(product_camera_details) if product_camera_details else np.nan)
        prod_page.append(page)

        print("+++++++++++++++++++++++++++++++ PAGE: ", page)
        print("prod_name: ", len(prod_name), "battery_details: ", len(battery_details), "camera_details: ", len(camera_details), "prod_discount: ", len(prod_discount))


    if page==int(tot_pages_count):
        break








range(1, 578)
============Page: 1
URL--------------- https://www.flipkart.com/search?q=smartphone&as=on&as-show=on&otracker=AS_Query_OrganicAutoSuggest_4_5_na_na_na&otracker1=AS_Query_OrganicAutoSuggest_4_5_na_na_na&as-pos=4&as-type=RECENT&suggestionId=smartphone&requestId=0d3788ae-4292-4222-b969-cf2f01a59412&as-searchtext=smart&page=1
all rows count: 10
+++++++++++++++++++++++++++++++ PAGE:  1
prod_name:  1 battery_details:  1 camera_details:  1 prod_discount:  1
+++++++++++++++++++++++++++++++ PAGE:  1
prod_name:  2 battery_details:  2 camera_details:  2 prod_discount:  2
+++++++++++++++++++++++++++++++ PAGE:  1
prod_name:  3 battery_details:  3 camera_details:  3 prod_discount:  3
+++++++++++++++++++++++++++++++ PAGE:  1
prod_name:  4 battery_details:  4 camera_details:  4 prod_discount:  4
+++++++++++++++++++++++++++++++ PAGE:  1
prod_name:  5 battery_details:  5 camera_details:  5 prod_discount:  5
+++++++++++++++++++++++++++++++ PAGE:  1
prod_name:  6 battery_details:  6 camera_d

In [15]:
# check if all counts are equal
print("=================")
print("prod_name: ", len(prod_name))
print("prod_brand: ", len(prod_brand))
print("prod_rating: ", len(prod_rating))
print("prod_price: ", len(prod_price))
print("prod_act_price: ", len(prod_act_price))
print("prod_page: ", len(prod_page))
print("prod_ram: ", len(prod_ram))
print("prod_storage: ", len(prod_storage))
print("battery_details: ", len(battery_details))
print("camera_details: ", len(camera_details))
print("prod_discount: ", len(prod_discount))

prod_name:  990
prod_brand:  990
prod_rating:  990
prod_price:  990
prod_act_price:  990
prod_page:  990
prod_ram:  990
prod_storage:  990
battery_details:  990
camera_details:  990
prod_discount:  990


In [16]:
prod_df = pd.DataFrame({
    "Product_Name": prod_name,
    "Brand": prod_brand,
    "Rating": prod_rating,
    "Price": prod_price,
    "Actual_Price": prod_act_price,
    "Page_number": prod_page,
    "RAM": prod_ram,
    "Storage": prod_storage,
    "Battery_Details": battery_details,
    "Camera_Details": camera_details,
    "Discount": prod_discount

})
prod_df.head(5)


,Product_Name,Brand,Rating,Price,Actual_Price,Page_number,RAM,Storage,Battery_Details,Camera_Details,Discount
0,"BOLTT EVO (Midnight Black, 64 GB)",BOLTT,4.3,"₹9,999","₹17,999",1,4 GB RAM,64 GB ROM,6000 mAh Lithium ion Battery,50MP + 2MP | 8MP Front Camera,44% off
1,"Mivi One 5G (Green, 128 GB)",Mivi,NaN,"₹12,999","₹21,999",1,4 GB RAM,128 GB ROM,6000 mAh Lithium Polymer Battery,50MP Rear Camera | 8MP Front Camera,40% off
2,"Samsung Galaxy F07 (Green, 64 GB)",Samsung,4.2,"₹11,999",NaN,1,4 GB RAM,64 GB ROM,5000 mAh Battery,50MP + 2MP | 8MP Front Camera,NaN
3,"BOLTT EVO (Berry Red, 64 GB)",BOLTT,4.3,"₹9,999","₹17,999",1,4 GB RAM,64 GB ROM,6000 mAh Lithium ion Battery,50MP + 2MP | 8MP Front Camera,44% off
4,"vivo T5 Lite 44W 5G (Twilight Shadow, 128 GB)",vivo,4.2,"₹20,499","₹33,999",1,4 GB RAM,128 GB ROM,6500 mAh Li-ion Battery,50MP + 0.08MP | 5MP Front Camera,39% off


In [17]:
prod_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 990 entries, 0 to 989
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Product_Name     990 non-null    str  
 1   Brand            990 non-null    str  
 2   Rating           862 non-null    str  
 3   Price            990 non-null    str  
 4   Actual_Price     893 non-null    str  
 5   Page_number      990 non-null    int64
 6   RAM              961 non-null    str  
 7   Storage          982 non-null    str  
 8   Battery_Details  962 non-null    str  
 9   Camera_Details   990 non-null    str  
 10  Discount         880 non-null    str  
dtypes: int64(1), str(10)
memory usage: 85.2 KB


In [18]:
prod_df.to_csv("flipkart_smartphones_scraped_data.csv", index=False)